In [ ]:
# %% [markdown]
# # Cross-evaluation notebook (A/B/C)

# %%
from pathlib import Path
import os, json, math, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# ---------- cohorts ----------
COHORTS = ["A", "B", "C"]

# ---------- point these to your saved datasets ----------
PKL_MAP = {
    "A": "v2_data_A.pkl",   # <<< EDIT
    "B": "v2_data_B.pkl",   # <<< EDIT
    "C": "v2_data_C.pkl",   # <<< EDIT
}

# ---------- project folders (same structure you used while training) ----------
ROOT        = Path("project")
DIR_MODELS  = ROOT / "models"
DIR_SPLITS  = ROOT / "splits"
DIR_CROSS   = ROOT / "cross"
DIR_FIGS    = ROOT / "figures"
for d in [ROOT, DIR_MODELS, DIR_SPLITS, DIR_CROSS, DIR_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- reproducibility ----------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# %% [markdown]
# ## 1) Model definition (paste/import your classes)

# %%
# ---- paste your model classes here OR import them ----
# from my_models import LightUNet_SE_ConvLSTM
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, t = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ConvSEBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1, dropout_prob=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(out_ch, momentum=0.05),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(out_ch, momentum=0.05),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock1D(out_ch)
        self.dropout = nn.Dropout(p=dropout_prob)
    def forward(self, x):
        x = self.conv(x)
        x = self.se(x)
        return self.dropout(x)

class TemporalConvMemory(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        self.depthwise = nn.Conv1d(channels, channels, kernel_size=kernel_size,
                                   padding=kernel_size // 2, groups=channels)
        self.pointwise = nn.Conv1d(channels, channels, kernel_size=1)
    def forward(self, x):
        mem = torch.relu(self.depthwise(x))
        mem = self.pointwise(mem)
        return mem + x

class LightUNet_SE_ConvLSTM(nn.Module):
    def __init__(self, input_channels=1, dropout_prob=0.2):
        super().__init__()
        self.enc1 = ConvSEBlock1D(input_channels, 32, dropout_prob=dropout_prob)
        self.down1 = nn.Conv1d(32, 32, kernel_size=3, stride=2, padding=1)
        self.enc2 = ConvSEBlock1D(32, 64, dropout_prob=dropout_prob)
        self.down2 = nn.Conv1d(64, 64, kernel_size=3, stride=2, padding=1)
        self.enc3 = ConvSEBlock1D(64, 128, dropout_prob=dropout_prob)
        self.down3 = nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1)
        self.temporal = TemporalConvMemory(128)
        self.up3 = nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2)
        self.dec3 = ConvSEBlock1D(192, 64, dropout_prob=dropout_prob)
        self.up2 = nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2)
        self.dec2 = ConvSEBlock1D(96, 32, dropout_prob=dropout_prob)
        self.up1 = nn.ConvTranspose1d(32, 16, kernel_size=2, stride=2)
        self.dec1 = ConvSEBlock1D(48, 16, dropout_prob=dropout_prob)
        self.out = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(16, 1))
    def forward(self, x):
        e1 = self.enc1(x); d1 = self.down1(e1)
        e2 = self.enc2(d1); d2 = self.down2(e2)
        e3 = self.enc3(d2); d3 = self.down3(e3)
        bottleneck = self.temporal(d3)
        u3 = nn.functional.interpolate(self.up3(bottleneck), size=e3.shape[-1])
        d3 = self.dec3(torch.cat([u3, e3], dim=1))
        u2 = nn.functional.interpolate(self.up2(d3), size=e2.shape[-1])
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = nn.functional.interpolate(self.up1(d2), size=e1.shape[-1])
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1).squeeze(1)

def load_model_for(cohort):
    ckpt = torch.load(DIR_MODELS / f"model_{cohort}.pt", map_location="cpu")
    # strip DataParallel prefix if present
    new_sd = {k.replace("module.", ""): v for k, v in ckpt.items()}
    model = LightUNet_SE_ConvLSTM(input_channels=1, dropout_prob=0.2)
    model.load_state_dict(new_sd, strict=True)
    model.to(device).eval()
    return model


In [ ]:
# %% [markdown]
# ## 2) Datasets & test loaders (per cohort)

# %%
class EEGBISDataset(Dataset):
    # keep numpy in RAM; convert to torch in __getitem__ (worker-friendly)
    def __init__(self, x_np, y_np, c_np, idxs):
        self.x = x_np[idxs].astype(np.float32)
        self.y = y_np[idxs].astype(np.float32)
        self.c = c_np[idxs].astype(np.int64)
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        eeg = torch.from_numpy(self.x[i]).unsqueeze(0)  # (1, T)
        bis = torch.tensor(self.y[i])
        return eeg, bis

def build_test_loader_for(cohort):
    # load dataset
    data = pd.read_pickle(PKL_MAP[cohort]) if PKL_MAP[cohort].endswith(".pkl") else joblib.load(PKL_MAP[cohort])
    x, y, c = data["x"], data["y"].astype(float), data["c"].astype(int)

    # load split
    with open(DIR_SPLITS / f"split_{cohort}.json", "r") as f:
        spl = json.load(f)
    test_caseids = set(spl["test_caseids"])

    # map caseid -> indices
    from collections import defaultdict
    cid2idx = defaultdict(list)
    for i, cid in enumerate(c):
        cid2idx[int(cid)].append(i)

    # gather test indices
    test_indices = [i for cid in test_caseids for i in cid2idx[int(cid)]]

    ds = EEGBISDataset(x, y, c, test_indices)
    # single-process loader (Windows/Jupyter safe)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=False)
    return loader

# prepare all test loaders
TEST_LOADERS = {coh: build_test_loader_for(coh) for coh in COHORTS}
for k,v in TEST_LOADERS.items():
    print(f"{k}: {len(v.dataset)} test segments")


In [ ]:
# %% [markdown]
# ## 3) Metrics & evaluation helpers

# %%
def concordance_ccc(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mu1, mu2 = np.mean(y_true), np.mean(y_pred)
    v1, v2 = np.var(y_true, ddof=1), np.var(y_pred, ddof=1)
    cov = np.mean((y_true - mu1)*(y_pred - mu2))
    return (2*cov) / (v1 + v2 + (mu1 - mu2)**2 + 1e-12)

@torch.no_grad()
def eval_on_loader(model, loader):
    y_true_all, y_pred_all = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        preds = model(xb).detach().cpu().numpy()
        y_true_all.append(yb.numpy())
        y_pred_all.append(preds)
    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    # reporting uses clipped preds
    y_pred_clip = np.clip(y_pred, 0.0, 100.0)
    mae  = mean_absolute_error(y_true, y_pred_clip)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred_clip))
    ccc  = concordance_ccc(y_true, y_pred_clip)
    return {"MAE": mae, "RMSE": rmse, "CCC": ccc}, (y_true, y_pred_clip)


In [ ]:
# %% [markdown]
# ## 4) Cross-evaluation matrix (Train→Eval)

# %%
# storage
mat_CCC  = pd.DataFrame(index=COHORTS, columns=COHORTS, dtype=float)
mat_MAE  = pd.DataFrame(index=COHORTS, columns=COHORTS, dtype=float)
mat_RMSE = pd.DataFrame(index=COHORTS, columns=COHORTS, dtype=float)

# also save pair-wise preds (Parquet→CSV fallback)
def save_pair_preds(train_c, eval_c, y_true, y_pred):
    df = pd.DataFrame({"y_true": y_true.astype(float), "y_pred": y_pred.astype(float)})
    out_parq = DIR_CROSS / f"preds_{train_c}_to_{eval_c}.parquet"
    out_csv  = DIR_CROSS / f"preds_{train_c}_to_{eval_c}.csv.gz"
    try:
        df.to_parquet(out_parq, index=False)
        print("Saved preds →", out_parq)
    except Exception as e:
        print(f"Parquet failed ({type(e).__name__}: {e}) — saving CSV.gz.")
        df.to_csv(out_csv, index=False, float_format="%.6f", compression="gzip")
        print("Saved preds →", out_csv)

for train_c in COHORTS:
    print(f"\n=== Loading model {train_c} ===")
    model = load_model_for(train_c)
    for eval_c in COHORTS:
        metrics, (yt, yp) = eval_on_loader(model, TEST_LOADERS[eval_c])
        mat_CCC.loc[train_c, eval_c]  = metrics["CCC"]
        mat_MAE.loc[train_c, eval_c]  = metrics["MAE"]
        mat_RMSE.loc[train_c, eval_c] = metrics["RMSE"]
        print(f"{train_c} → {eval_c} | CCC={metrics['CCC']:.3f} | MAE={metrics['MAE']:.3f} | RMSE={metrics['RMSE']:.3f}")
        save_pair_preds(train_c, eval_c, yt, yp)

# save matrices
mat_CCC.to_csv(DIR_CROSS / "cross_CCC.csv")
mat_MAE.to_csv(DIR_CROSS / "cross_MAE.csv")
mat_RMSE.to_csv(DIR_CROSS / "cross_RMSE.csv")
print("\nSaved matrices to", DIR_CROSS)


In [ ]:
# %% [markdown]
# ## 5) Δ vs in-domain baselines (negative = drop)

# %%
def delta_vs_in_domain(mat):
    # mat: rows=train, cols=eval
    deltas = pd.DataFrame(index=COHORTS, columns=[c for c in COHORTS if c != "A"]+["A"])  # keep order but it's arbitrary
    # compute (train→eval) − (eval→eval)
    for tr in COHORTS:
        for ev in COHORTS:
            deltas.loc[tr, ev] = mat.loc[tr, ev] - mat.loc[ev, ev]
    return deltas.astype(float)

delta_CCC  = delta_vs_in_domain(mat_CCC)
delta_MAE  = delta_vs_in_domain(mat_MAE)
delta_RMSE = delta_vs_in_domain(mat_RMSE)

delta_CCC.to_csv(DIR_CROSS / "cross_delta_CCC.csv")
delta_MAE.to_csv(DIR_CROSS / "cross_delta_MAE.csv")
delta_RMSE.to_csv(DIR_CROSS / "cross_delta_RMSE.csv")
print("Saved Δ tables.")


In [ ]:
# %% [markdown]
# ## 6) Heatmaps

# %%
def heatmap(mat, title, fname):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(mat.values.astype(float), aspect='auto')
    # annotate
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.iloc[i,j]:.2f}", ha='center', va='center',
                    fontsize=12, fontweight='bold', color='white' if mat.iloc[i,j] < np.nanmean(mat.values) else 'black')
    # ticks
    ax.set_xticks(range(len(mat.columns)))
    ax.set_yticks(range(len(mat.index)))
    ax.set_xticklabels(mat.columns, fontsize=14, fontweight='bold')
    ax.set_yticklabels(mat.index, fontsize=14, fontweight='bold')
    # titles
    ax.set_title(title, fontsize=18, fontweight='bold')
    ax.grid(False)
    fig.colorbar(im, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(DIR_CROSS / fname, dpi=200)
    plt.show()

heatmap(mat_CCC,  "CCC (Train→Eval)",  "cross_heatmap_CCC.png")
heatmap(mat_MAE,  "MAE (Train→Eval)",  "cross_heatmap_MAE.png")
heatmap(mat_RMSE, "RMSE (Train→Eval)", "cross_heatmap_RMSE.png")


In [ ]:
# %% [markdown]
# ## 7) Optional calibration plots for key pairs

# %%
def calibration_plot(y_true, y_pred, title, fname):
    bins = np.linspace(0, 100, 11)
    ids = np.digitize(y_true, bins) - 1
    pts = []
    for b in range(len(bins)-1):
        m = (ids == b)
        if m.sum() >= 20:
            pts.append((np.mean(y_true[m]), np.mean(y_pred[m]), m.sum()))
    pts = np.array(pts) if len(pts) else np.empty((0,3))
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot([0,100],[0,100],'k--',lw=1, label='Ideal')
    if len(pts):
        ax.scatter(pts[:,0], pts[:,1], s=30, alpha=0.9, label='Binned means')
        for xt, yp, n in pts: ax.text(xt, yp, f"{int(n)}", fontsize=12, fontweight='bold', ha='center', va='bottom')
    ax.set_title(title, fontsize=18, fontweight='bold')
    ax.set_xlabel("True BIS (bin mean)", fontsize=16, fontweight='bold')
    ax.set_ylabel("Pred BIS (mean)", fontsize=16, fontweight='bold')
    ax.tick_params(labelsize=14); 
    for lab in ax.get_xticklabels() + ax.get_yticklabels(): lab.set_fontweight('bold')
    ax.legend(fontsize=14); ax.grid(False); fig.tight_layout()
    fig.savefig(DIR_CROSS / fname, dpi=200); plt.show()

# Example pairs to visualize:
pairs = [("A","B"), ("A","C"), ("B","C"), ("C","B")]
for tr, ev in pairs:
    # load the saved pair preds (CSV.gz if Parquet not available)
    parq = DIR_CROSS / f"preds_{tr}_to_{ev}.parquet"
    csvg = DIR_CROSS / f"preds_{tr}_to_{ev}.csv.gz"
    if parq.exists():
        dfp = pd.read_parquet(parq)
    else:
        dfp = pd.read_csv(csvg)
    calibration_plot(dfp["y_true"].values, dfp["y_pred"].values,
                     title=f"Calibration: {tr}→{ev}", fname=f"calib_{tr}_to_{ev}.png")


In [ ]:
# %% [markdown]
# ## X) Complexity–performance measurement (batch=1, 10s window)

# %%
import time, math, io, gc, numpy as np, torch
import pandas as pd

# -------------------
# config
# -------------------
MODEL_NAME = "LightUNet_SE_ConvMem (base)"  # change per variant
SEQ_LEN = 1280  # 10 s @128 Hz
BATCH = 1
WARMUP = 50
ITERS  = 300

# choose a representative input
x_cpu = torch.randn(BATCH, 1, SEQ_LEN, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# If you already have a loaded model, reuse it; otherwise instantiate:
try:
    model
except NameError:
    model = LightUNet_SE_ConvLSTM(input_channels=1, dropout_prob=0.2)

# Ensure eval/inference mode
model.eval()
for p in model.parameters(): p.requires_grad_(False)

# -------------------
# helpers
# -------------------
def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def state_dict_size_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return len(buf.getvalue()) / (1024**2)

@torch.inference_mode()
def latency_ms(m, x, use_amp=False):
    # returns mean & p95 latency in milliseconds
    if x.device.type == "cuda":
        torch.cuda.synchronize()
    times = []
    for i in range(WARMUP + ITERS):
        t0 = time.perf_counter()
        if use_amp and x.device.type == "cuda":
            with torch.cuda.amp.autocast():
                _ = m(x)
        else:
            _ = m(x)
        if x.device.type == "cuda":
            torch.cuda.synchronize()
        if i >= WARMUP:
            times.append((time.perf_counter() - t0) * 1000.0)
    return float(np.mean(times)), float(np.percentile(times, 95))

@torch.inference_mode()
def peak_gpu_mem_mb(m, x):
    if x.device.type != "cuda":
        return None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _ = m(x)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / (1024**2)
    return float(peak)

def jit_trace(m, sample):
    # TorchScript (may speed CPU a bit)
    m = m.cpu().eval()
    with torch.inference_mode():
        ts = torch.jit.trace(m, sample)
        ts = torch.jit.optimize_for_inference(ts)
    return ts

def try_flops_with_thop(m, sample):
    # Optional FLOPs (install thop: pip install thop)
    try:
        from thop import profile
        macs, _ = profile(m, inputs=(sample,), verbose=False)
        # Some papers report GMACs; others report GFLOPs ≈ 2*GMACs
        gmacs = macs / 1e9
        gflops = 2 * gmacs
        return float(gmacs), float(gflops)
    except Exception as e:
        return None, None

# -------------------
# 1) Parameters & model size
# -------------------
params_total, params_train = count_params(model)
size_mb = state_dict_size_mb(model)

# -------------------
# 2) FLOPs / MACs (optional)
# -------------------
gmacs, gflops = try_flops_with_thop(model.cpu(), x_cpu)

# -------------------
# 3) Latency & memory
# -------------------
# CPU (FP32)
torch.set_num_threads(max(1, torch.get_num_threads()))  # keep current setting
m_cpu = model.cpu()
mean_cpu, p95_cpu = latency_ms(m_cpu, x_cpu, use_amp=False)

# TorchScript CPU (optional)
try:
    ts_model = jit_trace(model, x_cpu)
    mean_ts, p95_ts = latency_ms(ts_model, x_cpu, use_amp=False)
except Exception:
    mean_ts = p95_ts = None

# GPU (if available)
if torch.cuda.is_available():
    m_gpu = model.to(device)
    x_gpu = x_cpu.to(device, non_blocking=True)
    mean_gpu, p95_gpu = latency_ms(m_gpu, x_gpu, use_amp=False)         # FP32
    mean_gpu16, p95_gpu16 = latency_ms(m_gpu, x_gpu, use_amp=True)      # AMP FP16
    peak_mb = peak_gpu_mem_mb(m_gpu, x_gpu)
    # cleanup
    del x_gpu; gc.collect(); torch.cuda.empty_cache()
else:
    mean_gpu = p95_gpu = mean_gpu16 = p95_gpu16 = peak_mb = None

# -------------------
# 4) Collect & show
# -------------------
row = {
    "Model": MODEL_NAME,
    "Params (M)": round(params_total / 1e6, 3),
    "StateDict (MB)": round(size_mb, 2),
    "GMACs / window": (None if gmacs is None else round(gmacs, 3)),
    "GFLOPs / window": (None if gflops is None else round(gflops, 3)),
    "CPU FP32 mean (ms)": round(mean_cpu, 2),
    "CPU FP32 p95 (ms)": round(p95_cpu, 2),
    "CPU TorchScript mean (ms)": (None if mean_ts is None else round(mean_ts, 2)),
    "CPU TorchScript p95 (ms)": (None if p95_ts is None else round(p95_ts, 2)),
    "GPU FP32 mean (ms)": (None if mean_gpu is None else round(mean_gpu, 2)),
    "GPU FP32 p95 (ms)": (None if p95_gpu is None else round(p95_gpu, 2)),
    "GPU AMP FP16 mean (ms)": (None if mean_gpu16 is None else round(mean_gpu16, 2)),
    "GPU AMP FP16 p95 (ms)": (None if p95_gpu16 is None else round(p95_gpu16, 2)),
    "Peak GPU Mem (MB)": (None if peak_mb is None else round(peak_mb, 1)),
}
df_complexity = pd.DataFrame([row])
display(df_complexity)

# Optionally append to a CSV you’ll collect for all variants:
OUT = Path("project") / "metrics" / "complexity_summary.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
if OUT.exists():
    old = pd.read_csv(OUT)
    # avoid duplicate rows for same model name
    old = old[old["Model"] != MODEL_NAME]
    df_out = pd.concat([old, df_complexity], ignore_index=True)
else:
    df_out = df_complexity.copy()
df_out.to_csv(OUT, index=False)
print("Saved complexity summary →", OUT)
